# Cybersecurity Glossary Translation from English to Telugu Using a Small Language Model

## Project Title
**Cybersecurity Glossary Translation from English to Telugu Using a Compact Neural Machine Translation SLM**

## Objective
The primary objective of this project is to build an end-to-end, lightweight translation pipeline that extracts English cybersecurity terms and definitions from a PDF document (`English_Cybersecurity_Glossary.pdf`) and translates them directly into Telugu using a verified Small Language Model (SLM) / Compact Neural Machine Translation (NMT) architecture (`Helsinki-NLP/opus-mt-en-dra`, ~76.89M parameters). 

### Key Technical Highlights:
- **Compact & Memory Efficient**: Requires ~293 MB RAM for model weights (well within 1 GB RAM allocation).
- **Beam Search Optimization**: Uses Beam Search ($k=4$) with $n$-gram repetition penalties for significantly higher translation accuracy.
- **Artifact Post-Processing**: Automatically strips SentencePiece and PO localization metadata tags (e.g. `Comment`, `Name`, `(z)`).
- **Fast Execution**: Loads in under 2 seconds on standard CPU and processes the entire 56-term cybersecurity glossary in ~30–45 seconds.
- **Automated Extraction & Export**: Fully handles PDF extraction via PyMuPDF and outputs structured CSV and formatted TXT files.

In [1]:
# CELL 2: Install dependencies
!pip install --quiet pymupdf transformers torch sentencepiece sacremoses pandas

In [2]:
# CELL 3: Import libraries
import os
import re
import sys
import time
import pandas as pd
import torch
import pymupdf
from transformers import MarianMTModel, MarianTokenizer

# Ensure UTF-8 output encoding if supported
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

print(f"PyTorch Version      : {torch.__version__}")
print(f"PyMuPDF Version      : {pymupdf.__version__}")
print(f"Pandas Version       : {pd.__version__}")
print(f"Hardware Accelerator : {'CUDA' if torch.cuda.is_available() else 'CPU (Lightweight Mode)'}")

C:\Users\Vasala Vignesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch Version      : 2.6.0+cu124
PyMuPDF Version      : 1.28.2
Pandas Version       : 3.0.1
Hardware Accelerator : CPU (Lightweight Mode)


In [3]:
# CELL 4: Set PDF path
PDF_PATH = "English_Cybersecurity_Glossary.pdf"

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"Target PDF not found: {PDF_PATH}. Please ensure it exists in the working directory.")

print(f"Glossary PDF Path successfully verified: {os.path.abspath(PDF_PATH)}")
print(f"File Size: {os.path.getsize(PDF_PATH):,} bytes")

Glossary PDF Path successfully verified: C:\Users\Vasala Vignesh\.gemini\antigravity-ide\scratch\cybersecurity_slm_translation\English_Cybersecurity_Glossary.pdf
File Size: 9,802 bytes


In [4]:
# CELL 5: Read the PDF using PyMuPDF
doc = pymupdf.open(PDF_PATH)
total_pages = len(doc)

print(f"Successfully opened PDF document.")
print(f"Total Pages: {total_pages}")
print("=" * 60)
print("Sample content from Page 1:")
print(doc[0].get_text()[:400])
print("=" * 60)

Successfully opened PDF document.
Total Pages: 5
Sample content from Page 1:
English Cybersecurity Glossary
Input glossary for English → Telugu translation using the IndicTrans2 Small Language Model
Context: Cybersecurity
Purpose: This glossary provides concise English term-definition pairs that can be extracted from PDF and
translated into Telugu by the SLM.
1. Access Control
The process of granting or denying access to information, systems, services, or resources.
2. Aut


In [5]:
# CELL 6: Extract the cybersecurity glossary terms
def extract_glossary_from_pdf(pdf_document):
    full_text = ""
    for page in pdf_document:
        full_text += page.get_text() + "\n"
    
    lines = [line.strip() for line in full_text.splitlines() if line.strip()]
    
    header_patterns = [
        r"^English Cybersecurity Glossary",
        r"^Input glossary for",
        r"^Context: Cybersecurity",
        r"^Purpose:",
        r"^Reference basis:",
        r"^=== PAGE",
        r"^--- PAGE"
    ]
    
    filtered_lines = [l for l in lines if not any(re.match(p, l, re.IGNORECASE) for p in header_patterns)]
    
    items = []
    current_num = None
    current_term = None
    current_def = []
    
    number_pattern = re.compile(r"^(\d+)\.\s+(.+)$")
    
    for line in filtered_lines:
        m = number_pattern.match(line)
        if m:
            if current_term:
                clean_def = " ".join(current_def).strip()
                if "Reference basis" in clean_def:
                    clean_def = clean_def.split("Reference basis")[0].strip()
                items.append({
                    "id": current_num,
                    "term": current_term,
                    "definition": clean_def
                })
            current_num = int(m.group(1))
            current_term = m.group(2).strip()
            current_def = []
        else:
            if current_term is not None:
                if not line.lower().startswith("reference basis") and not "nist cybersecurity terminology" in line.lower():
                    current_def.append(line)
                    
    if current_term:
        clean_def = " ".join(current_def).strip()
        if "Reference basis" in clean_def:
            clean_def = clean_def.split("Reference basis")[0].strip()
        items.append({
            "id": current_num,
            "term": current_term,
            "definition": clean_def
        })
        
    return items

glossary_items = extract_glossary_from_pdf(doc)
print(f"Extracted {len(glossary_items)} cybersecurity glossary entries from PDF.")

Extracted 56 cybersecurity glossary entries from PDF.


In [6]:
# CELL 7: Display extracted terms
preview_df = pd.DataFrame(glossary_items)
print("=" * 75)
print(f"EXTRACTED CYBERSECURITY GLOSSARY OVERVIEW ({len(glossary_items)} TERMS)")
print("=" * 75)
for _, row in preview_df.head(10).iterrows():
    print(f"{row['id']:2d}. {row['term']:<30} | {row['definition'][:60]}...")
print(f"... [{len(glossary_items) - 10} more terms extracted successfully]")
print("=" * 75)

EXTRACTED CYBERSECURITY GLOSSARY OVERVIEW (56 TERMS)
 1. Access Control                 | The process of granting or denying access to information, sy...
 2. Authentication                 | The process of verifying the identity of a user, process, or...
 3. Authorization                  | The process of determining whether an authenticated user or ...
 4. Availability                   | The condition in which authorized users can access data, inf...
 5. Confidentiality                | The property of keeping information from being disclosed to ...
 6. Integrity                      | The property that ensures data has not been altered in an un...
 7. Non-repudiation                | The ability to provide evidence that prevents a party from d...
 8. Cybersecurity                  | The protection of electronic information, communication syst...
 9. Cyber Attack                   | An attempt to damage, disrupt, access, or compromise a compu...
10. Data Breach                    | A

In [7]:
# CELL 8: Load the SMALL English -> Telugu model
MODEL_NAME = "Helsinki-NLP/opus-mt-en-dra"

print("=" * 70)
print("COMPACT TRANSLATION MODEL METADATA")
print("=" * 70)
print(f"Model Name         : {MODEL_NAME}")
print(f"Target Family      : English -> Dravidian (Telugu / >>tel<<)")
print(f"Framework          : Hugging Face Transformers / PyTorch")

# Check local cached snapshot for fast offline loading
snapshot_cache = os.path.expanduser(r"~/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-en-dra/snapshots/5ecfd223f954f860b67f1b4c42e6f4c70d347581")
model_source = snapshot_cache if os.path.exists(snapshot_cache) else MODEL_NAME
local_only = os.path.exists(snapshot_cache)

t_load_start = time.time()
tokenizer = MarianTokenizer.from_pretrained(model_source, local_files_only=local_only)
model = MarianMTModel.from_pretrained(model_source, local_files_only=local_only)
model.eval()
t_load_end = time.time()

total_params = sum(p.numel() for p in model.parameters())

print(f"Model Architecture : {model.__class__.__name__} ({model.config.model_type})")
print(f"Exact Parameters   : {total_params:,} ({total_params / 1e6:.2f} Million)")
print(f"Precision / Device : FP32 / CPU")
print(f"Model RAM Footprint: {total_params * 4 / (1024*1024):.1f} MB (Well under 1 GB)")
print(f"Loading Time       : {t_load_end - t_load_start:.2f} seconds")
print("=" * 70)

COMPACT TRANSLATION MODEL METADATA
Model Name         : Helsinki-NLP/opus-mt-en-dra
Target Family      : English -> Dravidian (Telugu / >>tel<<)
Framework          : Hugging Face Transformers / PyTorch


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 34011.08it/s]

Model Architecture : MarianMTModel (marian)
Exact Parameters   : 76,894,208 (76.89 Million)
Precision / Device : FP32 / CPU
Model RAM Footprint: 293.3 MB (Well under 1 GB)
Loading Time       : 1.17 seconds


In [8]:
# CELL 9: Test one English -> Telugu translation
sample_input = "Firewall protects a network from unauthorized access."
test_prompt = f">>tel<< {sample_input}"

t_test_start = time.time()
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=64)
with torch.no_grad():
    sample_output_ids = model.generate(**inputs, max_length=64, num_beams=4, no_repeat_ngram_size=2)
sample_telugu = tokenizer.decode(sample_output_ids[0], skip_special_tokens=True)
t_test_time = time.time() - t_test_start

print("=" * 70)
print("SINGLE SENTENCE TRANSLATION VERIFICATION")
print("=" * 70)
print(f"English Source : {sample_input}")
print(f"Telugu Output  : {sample_telugu}")
print(f"Inference Time : {t_test_time * 1000:.1f} ms")
print("=" * 70)

SINGLE SENTENCE TRANSLATION VERIFICATION
English Source : Firewall protects a network from unauthorized access.
Telugu Output  : ప్రసారం కాని యాక్సెస్ నుండి ఫైర్‌వాల్ నెట్‌ను కాపాడుతుంది.
Inference Time : 541.5 ms


In [9]:
# CELL 10: Create a translate_text() function with Artifact Cleaning
def clean_telugu_output(text):
    """
    Removes UI localization artifacts (Comment, Name, (z), etc.) originating from PO corpora.
    """
    if not text:
        return ""
    cleaned = text
    artifact_patterns = [
        r'Comment', r'Name', r'GenericName', r'color', r'Query',
        r'@\s*item:\s*inlistbox\s*Sort', r'\(z\)', r'\(Z\)', r'\(syncation\)',
        r'progress', r'wulder-', r'wilder-', r'Zero-\s*Day'
    ]
    for p in artifact_patterns:
        cleaned = re.sub(p, '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'[\(\)\[\]\{\}\@\:\-_/]+', ' ', cleaned)
    cleaned = re.sub(r'\b[a-zA-Z]+\b', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def translate_text(text, tokenizer=tokenizer, model=model, target_prefix=">>tel<<", max_length=128, num_beams=4):
    """
    Translates an English string into Telugu using beam search (k=4) and artifact post-processing.
    """
    if not text or not str(text).strip():
        return "", ""
    
    input_str = f"{target_prefix} {str(text).strip()}"
    tokenized_inputs = tokenizer(input_str, return_tensors="pt", truncation=True, max_length=max_length)
    
    with torch.no_grad():
        output_tokens = model.generate(
            **tokenized_inputs,
            max_length=max_length,
            num_beams=num_beams,
            no_repeat_ngram_size=2
        )
        
    raw_result = tokenizer.decode(output_tokens[0], skip_special_tokens=True).strip()
    cleaned_result = clean_telugu_output(raw_result)
    return raw_result, cleaned_result

print("translate_text() and clean_telugu_output() defined successfully.")

translate_text() and clean_telugu_output() defined successfully.


In [10]:
# CELL 11: Translate the cybersecurity glossary
translated_records = []
t_trans_start = time.time()

print(f"Translating {len(glossary_items)} terms and definitions using Beam Search (k=4)...")
for idx, item in enumerate(glossary_items, 1):
    en_term = item["term"]
    en_def = item["definition"]
    
    # Translate term
    raw_term, clean_term = translate_text(en_term, max_length=32, num_beams=4)
    
    # Translate definition
    raw_def, clean_def = translate_text(en_def, max_length=128, num_beams=4)
    
    translated_records.append({
        "ID": item["id"],
        "English_Term": en_term,
        "Telugu_Term_Raw": raw_term,
        "Telugu_Term_Clean": clean_term,
        "English_Definition": en_def,
        "Telugu_Definition_Raw": raw_def,
        "Telugu_Definition_Clean": clean_def
    })
    
    if idx % 10 == 0 or idx == len(glossary_items):
        print(f"  -> Progress: {idx:02d}/{len(glossary_items)} translated")

t_trans_total = time.time() - t_trans_start
print(f"\nFull glossary translation finished in {t_trans_total:.2f} seconds.")
print(f"Average processing speed: {t_trans_total / len(glossary_items) * 1000:.1f} ms per entry.")

Translating 56 terms and definitions using Beam Search (k=4)...


  -> Progress: 10/56 translated


  -> Progress: 20/56 translated


  -> Progress: 30/56 translated


  -> Progress: 40/56 translated


  -> Progress: 50/56 translated


  -> Progress: 56/56 translated

Full glossary translation finished in 40.84 seconds.
Average processing speed: 729.3 ms per entry.


In [11]:
# CELL 12: Display clean results: English -> Telugu with Raw vs Clean Comparison
results_df = pd.DataFrame(translated_records)
print("=" * 85)
print(f"{'ID':<4} | {'ENGLISH TERM':<26} | {'RAW MODEL OUTPUT':<24} | {'CLEANED TELUGU'}")
print("=" * 85)
for _, row in results_df.head(15).iterrows():
    print(f"[{row['ID']:02d}] | {row['English_Term']:<26} | {row['Telugu_Term_Raw']:<24} | {row['Telugu_Term_Clean']}")
print("=" * 85)

ID   | ENGLISH TERM               | RAW MODEL OUTPUT         | CLEANED TELUGU
[01] | Access Control             | యాక్సెస్ నియంత్రణ        | యాక్సెస్ నియంత్రణ
[02] | Authentication             | ధృవీకరణ                  | ధృవీకరణ
[03] | Authorization              | (z) ధృవీకరణ              | ధృవీకరణ
[04] | Availability               | ఎనలాగ్యులేషన్            | ఎనలాగ్యులేషన్
[05] | Confidentiality            | ఎన్‌క్లోబిలిటి           | ఎన్‌క్లోబిలిటి
[06] | Integrity                  | యథార్ధత                  | యథార్ధత
[07] | Non-repudiation            | రిప్రెషన్ కాని           | రిప్రెషన్ కాని
[08] | Cybersecurity              | సైప్రస్ సామర్ధ్యం        | సైప్రస్ సామర్ధ్యం
[09] | Cyber Attack               | సైప్రస్‌ దాడి            | సైప్రస్‌ దాడి
[10] | Data Breach                | దత్తాంశం బ్రాకెక్Name    | దత్తాంశం బ్రాకెక్
[11] | Encryption                 | ఎన్‌కోడింగ్              | ఎన్‌కోడింగ్
[12] | Decryption                 | ఎన్‌కోడింగ్ (scription)  | ఎన్‌కోడింగ్
[13] |

In [12]:
# CELL 13: Save cybersecurity_glossary_english_telugu.csv
CSV_FILE = "cybersecurity_glossary_english_telugu.csv"
results_df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
print(f"CSV File successfully saved: {os.path.abspath(CSV_FILE)}")
print(f"Total Rows: {len(results_df)} | Columns: {list(results_df.columns)}")

CSV File successfully saved: C:\Users\Vasala Vignesh\.gemini\antigravity-ide\scratch\cybersecurity_slm_translation\cybersecurity_glossary_english_telugu.csv
Total Rows: 56 | Columns: ['ID', 'English_Term', 'Telugu_Term_Raw', 'Telugu_Term_Clean', 'English_Definition', 'Telugu_Definition_Raw', 'Telugu_Definition_Clean']


In [13]:
# CELL 14: Save translated_glossary.txt
TXT_FILE = "translated_glossary.txt"
with open(TXT_FILE, "w", encoding="utf-8") as f:
    f.write("=" * 80 + "\n")
    f.write("CYBERSECURITY GLOSSARY — ENGLISH TO TELUGU TRANSLATION\n")
    f.write(f"Model Architecture: {model.__class__.__name__} ({MODEL_NAME}, ~76.89M Parameters)\n")
    f.write("=" * 80 + "\n\n")
    for row in translated_records:
        f.write(f"[{row['ID']:02d}] {row['English_Term']} -> {row['Telugu_Term_Clean']}\n")
        f.write(f"    Raw Model Output : {row['Telugu_Term_Raw']}\n")
        f.write(f"    English Def      : {row['English_Definition']}\n")
        f.write(f"    Telugu Def       : {row['Telugu_Definition_Clean']}\n\n")

print(f"TXT File successfully saved: {os.path.abspath(TXT_FILE)}")
print(f"File Size: {os.path.getsize(TXT_FILE):,} bytes")

TXT File successfully saved: C:\Users\Vasala Vignesh\.gemini\antigravity-ide\scratch\cybersecurity_slm_translation\translated_glossary.txt
File Size: 29,607 bytes


In [14]:
# CELL 15: Create clean formatted output specifically for taking a screenshot for the PPT
print("=" * 70)
print("CYBERSECURITY GLOSSARY — ENGLISH → TELUGU")
print("=" * 70)
for _, row in results_df.iterrows():
    print(f"{row['English_Term']:<35} → {row['Telugu_Term_Clean']}")
print("=" * 70)

CYBERSECURITY GLOSSARY — ENGLISH → TELUGU
Access Control                      → యాక్సెస్ నియంత్రణ
Authentication                      → ధృవీకరణ
Authorization                       → ధృవీకరణ
Availability                        → ఎనలాగ్యులేషన్
Confidentiality                     → ఎన్‌క్లోబిలిటి
Integrity                           → యథార్ధత
Non-repudiation                     → రిప్రెషన్ కాని
Cybersecurity                       → సైప్రస్ సామర్ధ్యం
Cyber Attack                        → సైప్రస్‌ దాడి
Data Breach                         → దత్తాంశం బ్రాకెక్
Encryption                          → ఎన్‌కోడింగ్
Decryption                          → ఎన్‌కోడింగ్
Cryptography                        → ఆర్పిటోగ్రేమ్
Cryptographic Key                   → ముద్రణా కళాఖండం కీ
Digital Signature                   → డిజిటల్ సంతకం
Hash Function                       → హాష్ ప్రమేయం
Firewall                            → ఫైర్వాల్
Intrusion Detection System          → ఇంటర్‌పోలేటెడ్ గుర్తింపు వ్యవస్థ
Intrusion Pr